# Textile Industry Fake News Detection - Exploratory Data Analysis & Modeling

This notebook demonstrates the end-to-end NLP and ML workflow for classifying textile industry news articles as **REAL** or **POTENTIALLY MISLEADING (FAKE)**.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import custom NLP preprocessing
sys.path.append('..')
from preprocessing import preprocess_text

dataset_path = '../dataset/textile_news.csv'
df = pd.read_csv(dataset_path)
print(f"Total records: {len(df)}")
df.head()

In [ ]:
# Class distribution
df['label'].value_counts()

In [ ]:
# Category distribution
df['category'].value_counts()

In [ ]:
# Preprocess text
df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['cleaned_text'] = df['full_text'].apply(preprocess_text)
df[['full_text', 'cleaned_text', 'label']].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df['cleaned_text'].values
y = df['label'].values

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))